# 04 · Data quality gate

Runs after every load. If any check fails the notebook raises, the pipeline run fails, and the
semantic model is not refreshed with bad data. Results are appended to `gold_dq_results` for a
"data health" page in the report.

| Check | Why it matters |
|---|---|
| Key uniqueness | a duplicated PO item silently doubles spend |
| Referential integrity | facts pointing at missing dimension members drop out of slicers |
| Completeness | a missing exchange rate turns spend into `NULL` |
| Business rules | goods cannot be received before the PO exists; due dates follow invoice dates |
| Reconciliation to source | gold row counts tie back to the SAP extract |
| KPI reconciliation | gold KPIs match an independent implementation (`expected_metrics.json`, synthetic data only); in production, reconcile to SAP standard reports such as ME2N and the GR/IR clearing account |

In [ ]:
LANDING_PATH = "Files/landing/sap_p2p"
TABLE_FORMAT = "delta"
FAIL_ON_ERROR = True

In [ ]:
from pyspark.sql import functions as F

run_id = spark.table("bronze_load_log").agg(F.max("load_id")).first()[0]
results = []


def check(category: str, name: str, passed: bool, observed="", expected=""):
    results.append((run_id, category, name, bool(passed), str(observed), str(expected)))
    print(f"{'PASS' if passed else 'FAIL'}  {category:<16} {name}  (observed {observed}, expected {expected})")

In [ ]:
UNIQUE_KEYS = {
    "silver_purchase_order": ["purchase_order"],
    "silver_purchase_order_item": ["purchase_order", "purchase_order_item"],
    "silver_purchase_order_history": ["purchase_order", "purchase_order_item", "account_assignment_number",
                                      "event_type", "document_year", "document_number", "document_item"],
    "silver_supplier_invoice": ["invoice_key"],
    "gold_fact_po_item": ["po_item_key"],
    "gold_fact_supplier_invoice": ["invoice_key"],
    "gold_dim_date": ["date"],
    "gold_dim_supplier": ["supplier_id"],
    "gold_dim_material": ["material_id"],
    "gold_dim_material_group": ["material_group"],
    "gold_dim_plant": ["plant"],
    "gold_dim_purchasing_group": ["purchasing_group"],
}
for table, keys in UNIQUE_KEYS.items():
    duplicates = spark.table(table).groupBy(keys).count().filter("count > 1").count()
    check("uniqueness", f"{table} unique on ({', '.join(keys)})", duplicates == 0, duplicates, 0)

In [ ]:
FOREIGN_KEYS = [
    ("gold_fact_po_item", "supplier_id", "gold_dim_supplier", "supplier_id"),
    ("gold_fact_po_item", "material_id", "gold_dim_material", "material_id"),
    ("gold_fact_po_item", "material_group", "gold_dim_material_group", "material_group"),
    ("gold_fact_po_item", "plant", "gold_dim_plant", "plant"),
    ("gold_fact_po_item", "purchasing_group", "gold_dim_purchasing_group", "purchasing_group"),
    ("gold_fact_po_item", "po_date", "gold_dim_date", "date"),
    ("gold_fact_supplier_invoice", "supplier_id", "gold_dim_supplier", "supplier_id"),
    ("gold_fact_supplier_invoice", "posting_date", "gold_dim_date", "date"),
    ("gold_fact_supplier_invoice", "document_date", "gold_dim_date", "date"),
]
for fact, fk, dim, pk in FOREIGN_KEYS:
    orphans = (spark.table(fact).select(F.col(fk).alias("k")).distinct()
               .join(spark.table(dim).select(F.col(pk).alias("k")), "k", "left_anti").count())
    check("integrity", f"{fact}.{fk} → {dim}.{pk}", orphans == 0, orphans, 0)

In [ ]:
po = spark.table("gold_fact_po_item")
inv = spark.table("gold_fact_supplier_invoice")
NOT_NULL = [
    (po, "gold_fact_po_item", "net_amount_gbp"),
    (po, "gold_fact_po_item", "delivery_date"),
    (po, "gold_fact_po_item", "po_date"),
    (inv, "gold_fact_supplier_invoice", "gross_amount_gbp"),
    (inv, "gold_fact_supplier_invoice", "due_date"),
]
for df, table, column in NOT_NULL:
    nulls = df.filter(F.col(column).isNull()).count()
    check("completeness", f"{table}.{column} not null", nulls == 0, nulls, 0)

early_receipts = po.filter(F.col("first_gr_date") < F.col("po_date")).count()
check("business rule", "no goods receipt before the PO date", early_receipts == 0, early_receipts, 0)
bad_due = inv.filter(F.col("due_date") < F.col("document_date")).count()
check("business rule", "due date on or after invoice date", bad_due == 0, bad_due, 0)

source_items = spark.table("bronze_ekpo").count()
deleted_items = spark.table("silver_purchase_order_item").filter("is_deleted").count()
check("reconciliation", "PO items: gold + deleted = SAP extract", po.count() + deleted_items == source_items,
      po.count() + deleted_items, source_items)
source_invoices = spark.table("bronze_rbkp").count()
check("reconciliation", "invoices: gold = SAP extract", inv.count() == source_invoices, inv.count(), source_invoices)

In [ ]:
try:
    expected = (spark.read.option("multiLine", True).json(f"{LANDING_PATH}/expected_metrics.json")
                .first().asDict(recursive=True)["metrics"])
except Exception:  # no answer key: real SAP data
    expected = None
    print("No expected_metrics.json in the landing folder: KPI reconciliation skipped.")

if expected:
    observed = {r["metric"]: r["value"] for r in spark.table("gold_kpi_summary").collect()}
    for metric, want in expected.items():
        got = observed.get(metric)
        tolerance = 0.005 if metric.endswith("_gbp") else 1e-9
        passed = got is not None and abs(got - want) <= tolerance
        check("kpi", metric, passed, got, want)

In [ ]:
(spark.createDataFrame(results, "run_id string, category string, check_name string, passed boolean, "
                                "observed string, expected string")
 .withColumn("checked_at", F.current_timestamp())
 .write.mode("append").format(TABLE_FORMAT).saveAsTable("gold_dq_results"))

failures = [r for r in results if not r[3]]
print(f"\n{len(results) - len(failures)}/{len(results)} checks passed")
if failures and FAIL_ON_ERROR:
    raise AssertionError("Data quality gate failed: " + "; ".join(r[2] for r in failures))